# 02 - Entrenar KAN sobre embeddings (con PCA 95% varianza)

Este notebook:
1. Carga `embeddings.pt`.
2. Ajusta **PCA en train** con `n_components=0.95` (95% de varianza explicada).
3. Transforma val/test con ese mismo PCA.
4. Entrena KAN para clasificación binaria.
5. Selecciona mejor modelo por AUC de validación.
6. Calibra umbral por recall objetivo (MALIGNANT).
7. Guarda pesos/config y resumen en `Prueba_emmbedings/artifacts/`.

In [1]:
# Imports y configuración
from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.decomposition import PCA
from kan import KAN

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "Prueba_emmbedings" else Path.cwd().resolve()
ARTIFACTS_DIR = ROOT / "Prueba_emmbedings" / "artifacts"
EMB_PATH = ARTIFACTS_DIR / "embeddings.pt"
KAN_WEIGHTS_PATH = ARTIFACTS_DIR / "kan_head.pt"
KAN_CFG_PATH = ARTIFACTS_DIR / "kan_config.json"
SUMMARY_PATH = ARTIFACTS_DIR / "training_summary.json"
KAN_FEATURES_PATH = ARTIFACTS_DIR / "kan_features.pt"
PCA_LOADINGS_PATH = ARTIFACTS_DIR / "pca_loadings_top10.csv"
PCA_PAYLOAD_PATH = ARTIFACTS_DIR / "pca_payload.pt"

SEED = 42
KAN_HIDDEN = None  # Si None, se calcula automáticamente según input_dim
KAN_HIDDEN_SCALE = 0.25
KAN_HIDDEN_MIN = 16
KAN_HIDDEN_MAX = 128
KAN_GRID = 3
KAN_K = 3
LR = 1e-3
EPOCHS = 120
PATIENCE = 12
TARGET_RECALL = 0.80

# PCA opcional: si True, reduce dimensiones reteniendo el 95% de varianza.
USE_PCA = True
PCA_TARGET_VARIANCE = 0.95
PCA_TOP_LOADINGS_PER_COMPONENT = 10

DEVICE = torch.device("cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)

In [2]:
# Utilidades de métricas y umbral
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, precision_recall_curve

def safe_metrics(y_true, y_pred, y_prob=None):
    out = {
        "acc": float(accuracy_score(y_true, y_pred)),
        "recall_malignant": float(recall_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "precision_malignant": float(precision_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "f1_malignant": float(f1_score(y_true, y_pred, pos_label=1, zero_division=0)),
    }
    if y_prob is not None:
        out["auc"] = float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else float("nan")
    return out

def choose_threshold_by_recall(y_true, y_prob, target_recall=0.80):
    p, r, t = precision_recall_curve(y_true, y_prob)
    if len(t) == 0:
        return 0.5
    r_t = r[:-1]
    p_t = p[:-1]
    idx = np.where(r_t >= target_recall)[0]
    best_idx = int(np.argmax(r_t)) if len(idx) == 0 else int(idx[np.argmax(p_t[idx])])
    return float(t[best_idx])

def evaluate_logits(y_true_t, logits_t, thr=0.5):
    y_prob = torch.softmax(logits_t, dim=1)[:, 1].detach().cpu().numpy()
    y_true = y_true_t.detach().cpu().numpy().astype(int)
    y_pred = (y_prob >= thr).astype(int)
    m = safe_metrics(y_true, y_pred, y_prob)
    m["thr"] = float(thr)
    m["y_true"] = y_true
    m["y_pred"] = y_pred
    m["y_prob"] = y_prob
    return m

In [3]:
# Cargar embeddings y aplicar PCA opcional (fit SOLO en train)
data = torch.load(EMB_PATH, map_location="cpu")
Xtr_raw = data["X_train"].float()
ytr = data["y_train"].to(DEVICE).long()
Xva_raw = data["X_val"].float()
yva = data["y_val"].to(DEVICE).long()
Xte_raw = data["X_test"].float()
yte = data["y_test"].to(DEVICE).long()

pca = None
pca_input_dim = int(Xtr_raw.shape[1])
pca_explained_variance_sum = None

if USE_PCA:
    pca = PCA(n_components=PCA_TARGET_VARIANCE, svd_solver="full", random_state=SEED)
    Xtr_np = pca.fit_transform(Xtr_raw.numpy())
    Xva_np = pca.transform(Xva_raw.numpy())
    Xte_np = pca.transform(Xte_raw.numpy())

    Xtr = torch.tensor(Xtr_np, dtype=torch.float32, device=DEVICE)
    Xva = torch.tensor(Xva_np, dtype=torch.float32, device=DEVICE)
    Xte = torch.tensor(Xte_np, dtype=torch.float32, device=DEVICE)

    pca_input_dim = int(Xtr.shape[1])
    pca_explained_variance_sum = float(np.sum(pca.explained_variance_ratio_))

    print("Dim original:", Xtr_raw.shape[1])
    print("Dim PCA (95% var):", pca_input_dim)
    print("Varianza explicada acumulada:", pca_explained_variance_sum)
else:
    Xtr = Xtr_raw.to(DEVICE)
    Xva = Xva_raw.to(DEVICE)
    Xte = Xte_raw.to(DEVICE)
    print("PCA desactivado. Dim de entrada:", pca_input_dim)

if KAN_HIDDEN is None:
    hidden_dim = int(round(pca_input_dim * KAN_HIDDEN_SCALE))
    hidden_dim = max(KAN_HIDDEN_MIN, min(KAN_HIDDEN_MAX, hidden_dim))
else:
    hidden_dim = int(KAN_HIDDEN)

KAN_WIDTH = [pca_input_dim, hidden_dim, 2]
print("KAN hidden_dim:", hidden_dim)
print("KAN width:", KAN_WIDTH)

Dim original: 512
Dim PCA (95% var): 252
Varianza explicada acumulada: 0.9500230550765991
KAN hidden_dim: 63
KAN width: [252, 63, 2]


In [4]:
# Entrenamiento KAN
counts = torch.bincount(ytr, minlength=2).float().clamp_min(1)
class_weights = (len(ytr) / (2.0 * counts)).to(DEVICE)

model = KAN(width=KAN_WIDTH, grid=KAN_GRID, k=KAN_K, auto_save=False, seed=SEED).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_auc = -1.0
best_state = None
wait = 0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    logits_tr = model(Xtr)
    loss_tr = criterion(logits_tr, ytr)
    loss_tr.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        logits_va = model(Xva)

    tr_05 = evaluate_logits(ytr, logits_tr.detach(), thr=0.5)
    va_05 = evaluate_logits(yva, logits_va, thr=0.5)

    row = {
        "epoch": epoch,
        "tr_loss": float(loss_tr.item()),
        "tr_acc": float(tr_05["acc"]),
        "va_auc": float(va_05.get("auc", float("nan"))),
        "va_recall_mal": float(va_05["recall_malignant"]),
    }
    history.append(row)
    print(f"E{epoch:03d} | tr_loss={row['tr_loss']:.4f} tr_acc={row['tr_acc']:.4f} | va_auc={row['va_auc']:.4f} va_rec_mal={row['va_recall_mal']:.4f}")

    va_auc = float(va_05.get("auc", float("nan")))
    if va_auc > best_auc:
        best_auc = va_auc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print("Early stop.")
            break

model.load_state_dict(best_state)
print("Best val AUC:", best_auc)

E001 | tr_loss=0.6939 tr_acc=0.5257 | va_auc=0.5210 va_rec_mal=0.5823
E002 | tr_loss=0.6797 tr_acc=0.5641 | va_auc=0.5538 va_rec_mal=0.6386
E003 | tr_loss=0.6660 tr_acc=0.6121 | va_auc=0.5798 va_rec_mal=0.6546
E004 | tr_loss=0.6527 tr_acc=0.6484 | va_auc=0.6021 va_rec_mal=0.6667
E005 | tr_loss=0.6399 tr_acc=0.6747 | va_auc=0.6176 va_rec_mal=0.6667
E006 | tr_loss=0.6273 tr_acc=0.7011 | va_auc=0.6305 va_rec_mal=0.6667
E007 | tr_loss=0.6151 tr_acc=0.7201 | va_auc=0.6390 va_rec_mal=0.6747
E008 | tr_loss=0.6031 tr_acc=0.7417 | va_auc=0.6467 va_rec_mal=0.6867
E009 | tr_loss=0.5913 tr_acc=0.7521 | va_auc=0.6509 va_rec_mal=0.6827
E010 | tr_loss=0.5796 tr_acc=0.7637 | va_auc=0.6554 va_rec_mal=0.6827
E011 | tr_loss=0.5681 tr_acc=0.7771 | va_auc=0.6586 va_rec_mal=0.6707
E012 | tr_loss=0.5567 tr_acc=0.7862 | va_auc=0.6617 va_rec_mal=0.6546
E013 | tr_loss=0.5454 tr_acc=0.7952 | va_auc=0.6642 va_rec_mal=0.6506
E014 | tr_loss=0.5342 tr_acc=0.8026 | va_auc=0.6647 va_rec_mal=0.6627
E015 | tr_loss=0.523

In [5]:
# Métricas finales + guardado
model.eval()
with torch.no_grad():
    logits_va = model(Xva)
    logits_te = model(Xte)

m_va_05 = evaluate_logits(yva, logits_va, thr=0.5)
thr = choose_threshold_by_recall(m_va_05["y_true"], m_va_05["y_prob"], TARGET_RECALL)
m_va = evaluate_logits(yva, logits_va, thr=thr)
m_te = evaluate_logits(yte, logits_te, thr=thr)

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), KAN_WEIGHTS_PATH)

# Guardar features ya transformadas por PCA para el notebook de Sproud/simbólico.
torch.save(
    {
        "X_train": Xtr.detach().cpu(),
        "y_train": ytr.detach().cpu(),
        "X_val": Xva.detach().cpu(),
        "y_val": yva.detach().cpu(),
        "X_test": Xte.detach().cpu(),
        "y_test": yte.detach().cpu(),
    },
    KAN_FEATURES_PATH,
)

# Guardar artefactos de PCA para interpretabilidad (cargas por componente)
if USE_PCA and pca is not None:
    torch.save(
        {
            "mean": torch.tensor(pca.mean_, dtype=torch.float32),
            "components": torch.tensor(pca.components_, dtype=torch.float32),
            "explained_variance_ratio": torch.tensor(pca.explained_variance_ratio_, dtype=torch.float32),
            "n_components": int(pca.n_components_),
            "target_variance": float(PCA_TARGET_VARIANCE),
        },
        PCA_PAYLOAD_PATH,
    )

    rows = []
    comp = pca.components_  # [n_components, n_features]
    for c_idx in range(comp.shape[0]):
        w = comp[c_idx]
        top_idx = np.argsort(np.abs(w))[::-1][:PCA_TOP_LOADINGS_PER_COMPONENT]
        for rank, feat_idx in enumerate(top_idx, start=1):
            rows.append(
                {
                    "component": int(c_idx),
                    "rank_in_component": int(rank),
                    "original_dim": int(feat_idx),
                    "loading": float(w[feat_idx]),
                    "abs_loading": float(abs(w[feat_idx])),
                }
            )
    pca_loadings_df = pd.DataFrame(rows)
    pca_loadings_df.to_csv(PCA_LOADINGS_PATH, index=False)
    display(pca_loadings_df.head(20))

cfg = {
    "seed": SEED,
    "width": KAN_WIDTH,
    "grid": KAN_GRID,
    "k": KAN_K,
    "device": "cpu",
    "hidden_dim": int(hidden_dim),
    "hidden_rule": {
        "manual_hidden": KAN_HIDDEN,
        "scale": KAN_HIDDEN_SCALE,
        "min": KAN_HIDDEN_MIN,
        "max": KAN_HIDDEN_MAX,
    },
    "use_pca": bool(USE_PCA),
    "pca_target_variance": float(PCA_TARGET_VARIANCE) if USE_PCA else None,
    "pca_input_dim": int(pca_input_dim),
    "pca_explained_variance_sum": pca_explained_variance_sum,
}
with open(KAN_CFG_PATH, "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2)

summary = {
    "best_val_auc_epoch_training": best_auc,
    "selected_threshold_by_val_recall": thr,
    "target_recall": TARGET_RECALL,
    "val_metrics_at_threshold": {k: v for k, v in m_va.items() if k not in {"y_true", "y_pred", "y_prob"}},
    "test_metrics_at_threshold": {k: v for k, v in m_te.items() if k not in {"y_true", "y_pred", "y_prob"}},
    "history_tail": history[-10:],
    "model_capacity": {
        "input_dim": int(pca_input_dim),
        "hidden_dim": int(hidden_dim),
        "hidden_rule": {
            "manual_hidden": KAN_HIDDEN,
            "scale": KAN_HIDDEN_SCALE,
            "min": KAN_HIDDEN_MIN,
            "max": KAN_HIDDEN_MAX,
        },
    },
    "pca": {
        "enabled": bool(USE_PCA),
        "target_variance": float(PCA_TARGET_VARIANCE) if USE_PCA else None,
        "output_dim": int(pca_input_dim),
        "explained_variance_sum": pca_explained_variance_sum,
        "payload_path": str(PCA_PAYLOAD_PATH) if USE_PCA else None,
        "loadings_csv": str(PCA_LOADINGS_PATH) if USE_PCA else None,
    },
}

with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("KAN guardado en:", KAN_WEIGHTS_PATH)
print("Features KAN guardadas en:", KAN_FEATURES_PATH)
print("Config guardada en:", KAN_CFG_PATH)
print("Resumen guardado en:", SUMMARY_PATH)
if USE_PCA:
    print("PCA payload guardado en:", PCA_PAYLOAD_PATH)
    print("PCA loadings guardadas en:", PCA_LOADINGS_PATH)
print("Threshold:", thr)
print("Test metrics:", summary["test_metrics_at_threshold"])

,component,rank_in_component,original_dim,loading,abs_loading
0,0,1,406,0.120540,0.120540
1,0,2,290,0.108742,0.108742
2,0,3,275,0.108432,0.108432
3,0,4,94,0.107124,0.107124
4,0,5,30,0.104358,0.104358
5,0,6,313,0.099182,0.099182
6,0,7,33,0.095108,0.095108
7,0,8,62,0.093147,0.093147
8,0,9,15,0.092941,0.092941
9,0,10,364,0.092282,0.092282


KAN guardado en: G:\Cosas_programacion\Breast Cancer Interpretable-ml\Prueba_emmbedings\artifacts\kan_head.pt
Features KAN guardadas en: G:\Cosas_programacion\Breast Cancer Interpretable-ml\Prueba_emmbedings\artifacts\kan_features.pt
Config guardada en: G:\Cosas_programacion\Breast Cancer Interpretable-ml\Prueba_emmbedings\artifacts\kan_config.json
Resumen guardado en: G:\Cosas_programacion\Breast Cancer Interpretable-ml\Prueba_emmbedings\artifacts\training_summary.json
PCA payload guardado en: G:\Cosas_programacion\Breast Cancer Interpretable-ml\Prueba_emmbedings\artifacts\pca_payload.pt
PCA loadings guardadas en: G:\Cosas_programacion\Breast Cancer Interpretable-ml\Prueba_emmbedings\artifacts\pca_loadings_top10.csv
Threshold: 0.3571373224258423
Test metrics: {'acc': 0.5521327014218009, 'recall_malignant': 0.8333333333333334, 'precision_malignant': 0.47540983606557374, 'f1_malignant': 0.605427974947808, 'auc': 0.6401557285873192, 'thr': 0.3571373224258423}
